In [2]:
import pylab as plt
import numpy as np
from cartopy import crs as ccrs
from netCDF4 import Dataset
import xarray as xr
from scipy import ndimage
from scipy.optimize import minimize
from scipy.interpolate import interp1d
import h5py  


import sys as sys
import glob as glob
import datetime as datetime
sys.path.append('../src/')

import matplotlib.colors as mcolors
import cmocean
import cmocean.cm as cmo
lightcmap = cmo.thermal

from primary_microseisms import seismic_response_primary_sandwaves_spec


plt.rcParams.update({'font.size': 18,'savefig.facecolor':'white'})

In [3]:
wave_spectrum='/home/ltomaset/Documents/PROJECT/calcul_primaire/microseisms_LOPS/data/waves/CCI_WW3-GLOB-30M-Ekoundou_2023_spec.nc'
bottom_topography_spectrum='/home/ltomaset/Documents/PROJECT/calcul_primaire/microseisms_LOPS/data/bottom/spectrum_Ireland_shallow_rocks.bsp'
dpt=30     # adjusts water depth
CgR  =1800 # group speed of seismic waves, assumed constant

ds = Dataset(wave_spectrum, "r")
# list variables in the file
print(ds.variables.keys())
# read the frequency variable
freq = ds.variables["frequency"][:]
lat  = ds.variables["latitude"][0,0]
lon  = ds.variables["longitude"][0,0]
time_spec = ds.variables["time"][:]
date1=time_spec[0]
date2=time_spec[-1]
station='EDA'
dA=1E8*3600
Q=np.array(freq)*0+400
lono = 10.153427
lato = 3.778868
network = 'G'
startdate = '2019/05/04'
thresh1 = 1E-14
thresh2 = 8
thresh3 = 1.E-8
maxd = 2
maxdp = 1
r = np.maximum(0, (0.8 - 5 * freq))/6

#F_delta, Fx_delta, Fy_delta, Ef, freqp, df, times
F_delta,Fx_delta,Fy_delta,Ef,freqp,df,times1=seismic_response_primary_sandwaves_spec(wave_spectrum, bottom_topography_spectrum, dpt, lat,lon, CgR,  Q,  lono, lato, dA, station)
print('shape:',np.shape(F_delta))

dict_keys(['time', 'station', 'string40', 'station_name', 'longitude', 'latitude', 'frequency', 'frequency1', 'frequency2', 'direction', 'efth', 'dpt', 'wnd', 'wnddir', 'cur', 'curdir'])


TypeError: unsupported operand type(s) for ** or pow(): 'str' and 'int'

In [ ]:
#times,freqp,Ef_pris,Q,lato,lono=import_synthetic_mat('EDA_PDC_GLOBAL05_202301_TEST471_REF102040Q400_150_primary.mat')
times,freqp,Ef_pris,Q,lato,lono=import_synthetic_mat('../results/EDA_IRL_GLOBAL05_202301_CCI_WW3_Ekoundou_Q400_030_primary.mat')
times,freqp,Ef_pri2,Q,lato,lono=import_synthetic_mat('../results/EDA_IRL_GLOBAL05_202301_CCI_WW3_Akpo_Q400_200_primary.mat')

# Warning: data from Piero is probably PSD of acceleration ... 
file='../data/SpecEDA2023_psd.npz' # 'SpecEDA2023.npz'
date0, frq0, spectre0=read_sismo_npz(file)
print('FRQ:',np.shape(frq0),frq0[0:50])
print('SPEC:',np.shape(spectre0))
t1=np.datetime64('2023-07-02')
t2=np.datetime64('2023-08-19')
inds=np.where((date0 > t1) & (date0 < t2))[0]
print(date0[0],date0[1],date0[-1])
np.datetime64('2023-01-01')
fig, ax = plt.subplots(nrows=2, ncols=1,figsize=(15,10))
plt.subplots_adjust(left=0.05,bottom=0.07, top=0.92,wspace=0.12,right=0.99)
#im=ax[0].pcolormesh(10*np.log10(np.squeeze(spectre0)),cmap='viridis',rasterized=True) #,vmin=10, vmax=60)
print(np.shape(spectre0),np.shape(date0),np.shape(freqp))
im=ax[0].pcolormesh(date0[inds],frq0,10*np.log10(spectre0[inds,:].T),cmap='viridis',rasterized=True, shading='nearest',vmin=-200, vmax=-100)
_=plt.colorbar(im,ax=ax[0],label='', location='right',shrink=0.8)
_=ax[0].set_title('observed spectrum (dB)')
#_=ax[0].set_ylim([0.03,0.08])
_=ax[0].set_ylim([0.03,0.08])

print(np.shape(Ef_pris),np.shape(times),np.shape(freqp))
indsm=np.where((times > t1) & (times < t2))[0]
im=ax[1].pcolormesh(times[indsm],freqp[0:10],10*np.log10(Ef_pris[indsm,0:10].T),cmap='viridis',rasterized=True, shading='nearest',vmin=-200, vmax=-100)
_=plt.colorbar(im,ax=ax[1],label='', location='right',shrink=0.8)
_=ax[1].set_title('modeled spectrum (dB)')
_=ax[1].set_ylim([0.03,0.08])


fig, ax = plt.subplots(nrows=1, ncols=1,figsize=(15,5))
print(np.shape(F_delta),np.shape(times),np.shape(freqp))
indsm=np.where((times > t1) & (times < t2))[0]
im=ax.pcolormesh(times1[indsm],freqp[0:10],10*np.log10(F_delta[indsm,0:10].T),cmap='viridis',rasterized=True, shading='nearest',vmin=-200, vmax=-100)
_=plt.colorbar(im,ax=ax,label='', location='right',shrink=0.8)
_=ax.set_title('modeled with python (dB)')
_=ax.set_ylim([0.03,0.08])

In [ ]:
freqs=freqp
dfs=freqp*0.5*(1.1-1/1.1)
rs=dfs*0
print(lato,lono,freqs.shape)
# Set up paths
station='EDA' 
year=2023
yname = str(year)  # Replace with your desired year
#lato, lono, thresh1, thresh2, thresh3, maxd, rs, network, startdate=seismic.station_info(freqs, station)

ifmin = 1  # 0.1 
ifmax = 5 # 0.25

#ifmin = 5  # 0.1 
#ifmax = 8 # 0.25

print('Freq:',freqs[ifmin]*0.95,freqs[ifmax-1]*1.05)
# Assuming `date0` and `times` are datetime64 objects
base_date = np.datetime64('1970-01-01')  # You can use any base date, like '1970-01-01'

# Convert `date0` and `times` into seconds since the base date (this assumes `date0` and `times` are datetime64)
date0_seconds = (date0 - base_date) / np.timedelta64(1, 's')
times_seconds = (times - base_date) / np.timedelta64(1, 's')

# Now you can safely divide them by np.timedelta64(1, 's') if needed
delta_obs = seismic.compute_delta_obs(spectre0, frq0, freqs, ifmin, ifmax, date0_seconds, times_seconds,smooth=2,percentile=0)
delta_Eko, delta_ref = seismic.compute_delta_model(Ef_pris, Ef_pris, dfs, rs, ifmin, ifmax)
delta_Eko2, delta_ref = seismic.compute_delta_model(F_delta, F_delta, dfs, rs, ifmin, ifmax)
delta_Akp, delta_ref = seismic.compute_delta_model(Ef_pri2, Ef_pri2, dfs, rs, ifmin, ifmax)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1,figsize=(15,6))
P=60

t1=np.datetime64('2023-07-02')
t2=np.datetime64('2023-08-19')
inds=np.where((times > t1) & (times < t2))[0]

ax.plot(times[inds], delta_obs[inds], 'b-',alpha=0.5, label='measurements', linewidth=2)
ax.plot(times[inds], delta_Eko[inds] * P, 'k-', alpha=0.5,label='model - matlab: Ekoundou', linewidth=2)
ax.plot(times[inds], delta_Eko2[inds] , 'g--', alpha=0.5,label='model - python: Ekoundou', linewidth=3)
ax.plot(times[inds], delta_Akp[inds] * P, 'r-', alpha=0.5,label='model - matlab old: Akpo', linewidth=2)
#ax.plot(times, delta_PDC * P, 'k-', alpha=0.5,label='model: sandwaves', linewidth=2)
ax.set_xlabel('Time (Days)', fontsize=16)
ax.set_ylabel('Displacement (microns)', fontsize=16)
ax.set_ylim([0,0.08])
freqstring=f'{freqs[ifmin]*0.95:4.3f}'+'-'+f'{freqs[ifmax]*1.05:4.3f}'
ax.set_title('station '+station+' year '+yname+ ', frequencies:'+freqstring, fontsize=16)
plt.legend()
plt.grid(True)
        
plt.savefig('timeseries_freqs_newEDA'+freqstring+'.png')

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1,figsize=(15,6))
P=70
inds=np.where((times > t1) & (times < t2))[0]
#ax.plot(times[inds], delta_obs[inds], 'b-',alpha=0.5, label='measurements', linewidth=2)
ax.plot(times[inds], delta_Eko[inds] * P*3, 'k-', alpha=0.5,label='model: Ekoundou', linewidth=2)